# MitoChime Local Lab Runs\n\nRun this notebook from the **project root** after cloning the repo and installing dependencies.\n\nThis notebook covers:\n- environment and path checks\n- Gradient Boosting CV + held-out test\n- CNN 5-fold CV + held-out test\n- BiGRU 5-fold CV + held-out test\n- timing and summary tables\n

In [ ]:
import os, sys, json, time, shutil, platform\nfrom datetime import datetime\n\nPROJECT_ROOT = os.getcwd()\nSRC_ROOT = os.path.join(PROJECT_ROOT, 'src')\nDATA_ROOT = os.path.join(PROJECT_ROOT, 'data', 'processed')\nOUT_ROOT = os.path.join(PROJECT_ROOT, 'outputs_local')\n\nos.makedirs(OUT_ROOT, exist_ok=True)\nif SRC_ROOT not in sys.path:\n    sys.path.insert(0, SRC_ROOT)\n\nprint('PROJECT_ROOT =', PROJECT_ROOT)\nprint('SRC_ROOT =', SRC_ROOT)\nprint('DATA_ROOT =', DATA_ROOT)\nprint('OUT_ROOT =', OUT_ROOT)\nprint('Python =', sys.version)\nprint('Platform =', platform.platform())\n

In [ ]:
expected_files = [\n    os.path.join(DATA_ROOT, 'PAIR_train_noq.tsv'),\n    os.path.join(DATA_ROOT, 'PAIR_test_noq.tsv'),\n    os.path.join(DATA_ROOT, 'PAIR_train_seq_L150.tsv'),\n    os.path.join(DATA_ROOT, 'PAIR_test_seq_L150.tsv'),\n]\nfor p in expected_files:\n    print(os.path.exists(p), p)\n\nfor k in range(5):\n    print(os.path.exists(os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{k}_train_seq.tsv')),\n          os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{k}_train_seq.tsv'))\n    print(os.path.exists(os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{k}_val_seq.tsv')),\n          os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{k}_val_seq.tsv'))\n

## Gradient Boosting\nThis reproduces the original ML workflow: 5-fold Stratified CV on training set, then fit on full training set, then evaluate on held-out test set.\n

In [ ]:
import numpy as np\nimport pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.model_selection import StratifiedKFold, cross_validate\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.ensemble import GradientBoostingClassifier\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix\nimport joblib\n\nRANDOM_STATE = 42\ntrain_tab = os.path.join(DATA_ROOT, 'PAIR_train_noq.tsv')\ntest_tab = os.path.join(DATA_ROOT, 'PAIR_test_noq.tsv')\n\ntrain_df = pd.read_csv(train_tab, sep='\t')\ntest_df = pd.read_csv(test_tab, sep='\t')\n\nX_train = train_df.drop(columns=['label']).select_dtypes(include=['number']).copy()\nX_test = test_df.drop(columns=['label']).select_dtypes(include=['number']).copy()\ny_train = train_df['label'].copy()\ny_test = test_df['label'].copy()\n\nrun_name = 'gradient_boosting_pair_noq_local'\nrun_dir = os.path.join(OUT_ROOT, 'reports', run_name)\nmodel_dir = os.path.join(OUT_ROOT, 'models', run_name)\nos.makedirs(run_dir, exist_ok=True)\nos.makedirs(model_dir, exist_ok=True)\n\npipe = Pipeline([\n    ('imputer', SimpleImputer(strategy='median')),\n    ('scaler', StandardScaler()),\n    ('clf', GradientBoostingClassifier(\n        n_estimators=300,\n        learning_rate=0.1,\n        max_depth=3,\n        subsample=0.9,\n        random_state=42,\n    )),\n])\n\ncv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n\nstart = time.time()\nstart_iso = datetime.now().isoformat()\ncv_results = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['accuracy', 'f1'], return_train_score=False, n_jobs=-1)\npipe.fit(X_train, y_train)\nelapsed = time.time() - start\nend_iso = datetime.now().isoformat()\n\ny_pred = pipe.predict(X_test)\ny_prob = pipe.predict_proba(X_test)[:, 1]\n\nmetrics = {\n    'model': 'gradient_boosting',\n    'cv_accuracy_mean': float(np.mean(cv_results['test_accuracy'])),\n    'cv_accuracy_std': float(np.std(cv_results['test_accuracy'])),\n    'cv_f1_mean': float(np.mean(cv_results['test_f1'])),\n    'cv_f1_std': float(np.std(cv_results['test_f1'])),\n    'test_accuracy': float(accuracy_score(y_test, y_pred)),\n    'test_precision': float(precision_score(y_test, y_pred, zero_division=0)),\n    'test_recall': float(recall_score(y_test, y_pred, zero_division=0)),\n    'test_f1': float(f1_score(y_test, y_pred, zero_division=0)),\n    'test_roc_auc': float(roc_auc_score(y_test, y_prob)),\n    'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),\n    'n_train': int(len(X_train)),\n    'n_test': int(len(X_test)),\n    'n_features': int(X_train.shape[1]),\n}\n\ntiming = {\n    'run_name': run_name,\n    'platform': 'local_lab',\n    'start_time': start_iso,\n    'end_time': end_iso,\n    'elapsed_seconds': elapsed,\n    'elapsed_minutes': elapsed / 60,\n}\n\njoblib.dump(pipe, os.path.join(model_dir, 'gradient_boosting.joblib'))\nwith open(os.path.join(run_dir, 'gradient_boosting_metrics.json'), 'w') as f:\n    json.dump(metrics, f, indent=2)\nwith open(os.path.join(OUT_ROOT, f'timing_{run_name}.json'), 'w') as f:\n    json.dump(timing, f, indent=2)\n\nprint(json.dumps(metrics, indent=2))\nprint('Elapsed minutes:', elapsed / 60)\n

## Helper for deep-learning subprocess runs\n

In [ ]:
def run_subprocess(cmd, run_name):\n    os.makedirs(os.path.join(OUT_ROOT, 'models', run_name), exist_ok=True)\n    os.makedirs(os.path.join(OUT_ROOT, 'reports', run_name), exist_ok=True)\n    env = os.environ.copy()\n    env['PYTHONPATH'] = SRC_ROOT\n    start = time.time()\n    start_iso = datetime.now().isoformat()\n    result = subprocess.run(cmd, env=env, capture_output=True, text=True)\n    elapsed = time.time() - start\n    end_iso = datetime.now().isoformat()\n    timing = {\n        'run_name': run_name,\n        'platform': 'local_lab',\n        'start_time': start_iso,\n        'end_time': end_iso,\n        'elapsed_seconds': elapsed,\n        'elapsed_minutes': elapsed / 60,\n        'return_code': result.returncode,\n    }\n    with open(os.path.join(OUT_ROOT, f'timing_{run_name}.json'), 'w') as f:\n        json.dump(timing, f, indent=2)\n    print('Return code:', result.returncode)\n    print('STDOUT tail:\n', result.stdout[-8000:])\n    print('STDERR tail:\n', result.stderr[-4000:])\n    print('Elapsed minutes:', elapsed / 60)\n    return result\n

## CNN 5-fold CV\n

In [ ]:
cnn_cv_runs = []\nfor fold in range(5):\n    run_name = f'cnn_cv30_fold{fold}_L150_seed42_local'\n    train_tsv = os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{fold}_train_seq.tsv')\n    val_tsv = os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{fold}_val_seq.tsv')\n    cmd = [\n        sys.executable, '-m', 'mitochime.deep_learning.train_deep',\n        '--mode', 'cnn',\n        '--train-tsv', train_tsv,\n        '--test-tsv', val_tsv,\n        '--L', '150',\n        '--epochs', '30',\n        '--batch', '128',\n        '--lr', '0.001',\n        '--seed', '42',\n        '--select-best-by', 'f1',\n        '--weight-decay', '1e-4',\n        '--out-dir', os.path.join(OUT_ROOT, 'models', run_name),\n        '--reports-dir', os.path.join(OUT_ROOT, 'reports', run_name),\n        '--save-predictions',\n    ]\n    print('=' * 80)\n    print('RUNNING', run_name)\n    res = run_subprocess(cmd, run_name)\n    cnn_cv_runs.append((fold, res.returncode))\ncnn_cv_runs\n

## CNN held-out test\n

In [ ]:
run_name = 'cnn_holdout30_L150_seed42_local'\ncmd = [\n    sys.executable, '-m', 'mitochime.deep_learning.train_deep',\n    '--mode', 'cnn',\n    '--train-tsv', os.path.join(DATA_ROOT, 'PAIR_train_seq_L150.tsv'),\n    '--test-tsv', os.path.join(DATA_ROOT, 'PAIR_test_seq_L150.tsv'),\n    '--L', '150',\n    '--epochs', '30',\n    '--batch', '128',\n    '--lr', '0.001',\n    '--seed', '42',\n    '--select-best-by', 'f1',\n    '--weight-decay', '1e-4',\n    '--out-dir', os.path.join(OUT_ROOT, 'models', run_name),\n    '--reports-dir', os.path.join(OUT_ROOT, 'reports', run_name),\n    '--save-predictions',\n]\nrun_subprocess(cmd, run_name)\n

## BiGRU 5-fold CV\n

In [ ]:
bigru_cv_runs = []\nfor fold in range(5):\n    run_name = f'bigru_cv30_fold{fold}_L150_seed42_local'\n    train_tsv = os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{fold}_train_seq.tsv')\n    val_tsv = os.path.join(DATA_ROOT, 'cv_seq_L150_seed42', f'fold{fold}_val_seq.tsv')\n    cmd = [\n        sys.executable, '-m', 'mitochime.deep_learning.train_deep',\n        '--mode', 'rnn_kmer_gru',\n        '--train-tsv', train_tsv,\n        '--test-tsv', val_tsv,\n        '--L', '150',\n        '--k', '4',\n        '--L-kmers', '147',\n        '--embed-dim', '64',\n        '--hidden', '256',\n        '--rnn-layers', '1',\n        '--bidirectional',\n        '--pool', 'last',\n        '--epochs', '30',\n        '--batch', '128',\n        '--lr', '0.001',\n        '--seed', '42',\n        '--select-best-by', 'f1',\n        '--weight-decay', '1e-4',\n        '--out-dir', os.path.join(OUT_ROOT, 'models', run_name),\n        '--reports-dir', os.path.join(OUT_ROOT, 'reports', run_name),\n        '--save-predictions',\n    ]\n    print('=' * 80)\n    print('RUNNING', run_name)\n    res = run_subprocess(cmd, run_name)\n    bigru_cv_runs.append((fold, res.returncode))\nbigru_cv_runs\n

## BiGRU held-out test\n

In [ ]:
run_name = 'bigru_holdout30_L150_seed42_local'\ncmd = [\n    sys.executable, '-m', 'mitochime.deep_learning.train_deep',\n    '--mode', 'rnn_kmer_gru',\n    '--train-tsv', os.path.join(DATA_ROOT, 'PAIR_train_seq_L150.tsv'),\n    '--test-tsv', os.path.join(DATA_ROOT, 'PAIR_test_seq_L150.tsv'),\n    '--L', '150',\n    '--k', '4',\n    '--L-kmers', '147',\n    '--embed-dim', '64',\n    '--hidden', '256',\n    '--rnn-layers', '1',\n    '--bidirectional',\n    '--pool', 'last',\n    '--epochs', '30',\n    '--batch', '128',\n    '--lr', '0.001',\n    '--seed', '42',\n    '--select-best-by', 'f1',\n    '--weight-decay', '1e-4',\n    '--out-dir', os.path.join(OUT_ROOT, 'models', run_name),\n    '--reports-dir', os.path.join(OUT_ROOT, 'reports', run_name),\n    '--save-predictions',\n]\nrun_subprocess(cmd, run_name)\n

## Summary tables\n

In [ ]:
def summarize_cv(prefix, metrics_filename):\n    rows = []\n    for fold in range(5):\n        run_name = f'{prefix}_cv30_fold{fold}_L150_seed42_local'\n        metrics_path = os.path.join(OUT_ROOT, 'reports', run_name, metrics_filename)\n        timing_path = os.path.join(OUT_ROOT, f'timing_{run_name}.json')\n        row = {'fold': fold}\n        if os.path.exists(metrics_path):\n            with open(metrics_path) as f:\n                metrics = json.load(f)\n            row.update({\n                'accuracy': metrics.get('accuracy'),\n                'precision': metrics.get('precision'),\n                'recall': metrics.get('recall'),\n                'f1': metrics.get('f1'),\n                'roc_auc': metrics.get('roc_auc'),\n                'test_loss': metrics.get('test_loss'),\n            })\n        if os.path.exists(timing_path):\n            with open(timing_path) as f:\n                timing = json.load(f)\n            row['elapsed_minutes'] = timing.get('elapsed_minutes')\n        rows.append(row)\n    df = pd.DataFrame(rows)\n    return df\n\ncnn_cv = summarize_cv('cnn', 'cnn_metrics.json')\nbigru_cv = summarize_cv('bigru', 'rnn_kmer_gru_metrics.json')\n\nprint('CNN CV')\ndisplay(cnn_cv)\ndisplay(cnn_cv.mean(numeric_only=True).to_frame().T)\n\nprint('BiGRU CV')\ndisplay(bigru_cv)\ndisplay(bigru_cv.mean(numeric_only=True).to_frame().T)\n

In [ ]:
archive_base = os.path.join(PROJECT_ROOT, 'outputs_local_archive')\nshutil.make_archive(archive_base, 'zip', OUT_ROOT)\nprint('Created:', archive_base + '.zip')\n